In [34]:
import pandas as pd

def decode_bytes(hex_parts):
    """Decode hex strings, handling escape sequences: 4d 6a -> 4a, 4d 6d -> 4d"""
    result = []
    i = 0
    while i < len(hex_parts):
        val = int(hex_parts[i], 16)
        if val == 0x4d and i + 1 < len(hex_parts):
            next_val = int(hex_parts[i + 1], 16)
            if next_val == 0x6a:
                result.append(0x4a)
                i += 2
                continue
            elif next_val == 0x6d:
                result.append(0x4d)
                i += 2
                continue
        result.append(val)
        i += 1
    return result

# Load and parse the dump file
data = []

with open('bear2.dump', 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        
        # Decode escape sequences
        bytes_decoded = decode_bytes(parts)
        
        if len(bytes_decoded) < 7:
            continue
        
        # Ignore first value (bytes_decoded[0])
        # Concatenate bytes 2-4 into 24-bit offset (big-endian)
        offset = (bytes_decoded[1] << 16) | (bytes_decoded[2] << 8) | bytes_decoded[3]
        
        # Concatenate bytes 5-7 into 24-bit length (big-endian)
        length = (bytes_decoded[4] << 16) | (bytes_decoded[5] << 8) | bytes_decoded[6]
        
        # Concatenate bytes 8-9 into 16-bit timestamp if available
        timestamp = None
        if len(bytes_decoded) >= 9:
            timestamp = (bytes_decoded[7] << 8) | bytes_decoded[8]
        
        data.append({'offset': offset, 'length': length, 'timestamp': timestamp})

df = pd.DataFrame(data)

# Timestamp unit in milliseconds
TS_UNIT_MS = 1.31072

# Compute timestamp difference to next row
df['ts_diff'] = df['timestamp'].shift(-1) - df['timestamp']

# Add columns with millisecond values
df['timestamp_ms'] = df['timestamp'] * TS_UNIT_MS
df['ts_diff_ms'] = df['ts_diff'] * TS_UNIT_MS

print(f"Loaded {len(df)} records")
print(f"Records with timestamp: {df['timestamp'].notna().sum()}")
print(f"Records without timestamp: {df['timestamp'].isna().sum()}")
df.head(20)

Loaded 58194 records
Records with timestamp: 53201
Records without timestamp: 4993


,offset,length,timestamp,ts_diff
0,240,4,57391.0,0.0
1,528,8,57391.0,0.0
2,2011460,4,57391.0,4.0
3,2011464,68,57395.0,33.0
4,2011532,80,57428.0,27.0
5,2011612,76,57455.0,28.0
6,2011688,76,57483.0,NaN
7,2011764,74,NaN,NaN
8,2011838,82,57539.0,26.0
9,2011920,66,57565.0,28.0


In [4]:
# Load flash.bin as byte array
with open('flash.bin', 'rb') as f:
    flash = bytearray(f.read())

print(f"Loaded {len(flash)} bytes ({len(flash)/1024:.1f} KB)")

Loaded 4194304 bytes (4096.0 KB)


In [5]:
def hexdump(dataframe, flash_data, max_rows=None):
    """Dump dataframe records from flash in hexdump -C style.
    
    Args:
        dataframe: DataFrame with 'offset' and 'length' columns
        flash_data: bytearray containing flash contents
        max_rows: limit number of records to dump (None for all)
    """
    rows = dataframe if max_rows is None else dataframe.head(max_rows)
    
    for idx, row in rows.iterrows():
        offset = int(row['offset'])
        length = int(row['length'])
        print(f"=== Record {idx}: offset=0x{offset:06x}, length={length} ===")
        
        data = flash_data[offset:offset + length]
        
        for i in range(0, len(data), 16):
            chunk = data[i:i + 16]
            addr = offset + i
            
            # Hex part: two groups of 8 bytes
            hex_left = ' '.join(f'{b:02x}' for b in chunk[:8])
            hex_right = ' '.join(f'{b:02x}' for b in chunk[8:])
            hex_part = f'{hex_left:<23}  {hex_right:<23}'
            
            # ASCII part
            ascii_part = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
            
            print(f'{addr:08x}  {hex_part} |{ascii_part}|')
        print()

# Example: dump first 3 records
hexdump(df, flash, max_rows=3)

=== Record 0: offset=0x0000f0, length=4 ===
000000f0  1f b2 a4 00                                      |....|

=== Record 1: offset=0x000210, length=8 ===
00000210  59 b3 7f b3 7f d3 6b b3                          |Y.....k.|

=== Record 2: offset=0x1eb144, length=4 ===
001eb144  aa be ea 08                                      |....|



In [9]:
def filter_by_length(dataframe, include_lengths):
    """Filter to keep only rows with specific lengths.
    
    Args:
        dataframe: DataFrame with 'length' column
        include_lengths: single length or list of lengths to include
    
    Returns:
        Filtered DataFrame
    """
    if isinstance(include_lengths, (int, float)):
        include_lengths = [include_lengths]
    return dataframe[dataframe['length'].isin(include_lengths)]

# Example: keep only records with length 4 or 8
df_filtered = filter_by_length(df, [4, 8])
print(f"Original: {len(df)} records")
print(f"Filtered: {len(df_filtered)} records")
df_filtered.head(21)

Original: 58194 records
Filtered: 21 records


,offset,length,timestamp
0,240,4,57391.0
1,528,8,57391.0
2,2011460,4,57391.0
1646,2137982,8,37164.0
11741,2915916,8,53411.0
11746,2915948,8,53549.0
11748,520,8,53653.0
11749,960354,4,53654.0
25394,2011444,8,36713.0
25396,512,8,36817.0


In [10]:
filter_by_length(df, [8])

,offset,length,timestamp
1,528,8,57391.0
1646,2137982,8,37164.0
11741,2915916,8,53411.0
11746,2915948,8,53549.0
11748,520,8,53653.0
25394,2011444,8,36713.0
25396,512,8,36817.0
37854,960338,8,52646.0
37856,536,8,52750.0
46441,3577062,8,27322.0


In [15]:
def filter_preceding(dataframe, target_lengths):
    """Filter to keep only rows that precede rows with specific lengths.
    
    Args:
        dataframe: DataFrame with 'length' column
        target_lengths: single length or list of lengths to find predecessors of
    
    Returns:
        DataFrame containing rows that immediately precede target length rows
    """
    if isinstance(target_lengths, (int, float)):
        target_lengths = [target_lengths]
    
    # Find indices where length matches target
    target_mask = dataframe['length'].isin(target_lengths)
    target_indices = dataframe.index[target_mask]
    
    # Get the preceding indices (shift back by 1 in position)
    positions = [dataframe.index.get_loc(idx) - 1 for idx in target_indices]
    preceding_indices = [dataframe.index[pos] for pos in positions if pos >= 0]
    
    return dataframe.loc[preceding_indices]

# Example: find rows that precede rows with length 4
df_preceding = filter_preceding(df, 4)
print(f"Found {len(df_preceding)} rows preceding length-4 records")
df_preceding.head(10)

Found 5 rows preceding length-4 records


,offset,length,timestamp
1,528,8,57391.0
11748,520,8,53653.0
25396,512,8,36817.0
37856,536,8,52750.0
46447,528,8,27536.0


In [38]:
df.iloc[11700:11750]

,offset,length,timestamp,ts_diff
11700,2912714,70,52283.0,29.0
11701,2912784,110,52312.0,26.0
11702,2912894,68,52338.0,28.0
11703,2912962,68,52366.0,27.0
11704,2913030,70,52393.0,29.0
11705,2913100,84,52422.0,28.0
11706,2913184,92,52450.0,26.0
11707,2913276,70,52476.0,28.0
11708,2913346,70,52504.0,28.0
11709,2913416,96,52532.0,NaN


|Segment|begin|end|
|-------|-|-|
|1|0x001eb144|0x002c7e7a|
|2|0x00ee762|0x01eb142|
|3|0x000230|0x00ea760|
|4|0x002c7e7c|0x00369506|

